In [ ]:

# 1. Instalación de dependencias (En Colab esto es rápido y sin errores)
!pip install -q transformers spacy scikit-learn matplotlib seaborn
!python -m spacy download es_core_news_sm

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 DÍA 23: NLP APLICADO - HUGGING FACE, LDA, NER")
print("="*70)

# 2. Importar Librerías
print("\n Importando librerías...")

import spacy
from transformers import pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

print("🤖 Cargando spaCy en español...")
nlp_es = spacy.load('es_core_news_sm')

# Diccionario para descripciones de NER (evita warnings de spacy.explain)
NER_DESCRIPTIONS = {
    'PERSON': 'Persona', 'NORP': 'Nacionalidad/Grupo', 'FAC': 'Instalación',
    'ORG': 'Organización', 'GPE': 'País/Ciudad', 'LOC': 'Lugar',
    'PRODUCT': 'Producto', 'EVENT': 'Evento', 'WORK_OF_ART': 'Obra de arte',
    'LAW': 'Ley', 'LANGUAGE': 'Idioma', 'DATE': 'Fecha', 'TIME': 'Hora',
    'PERCENT': 'Porcentaje', 'MONEY': 'Dinero', 'QUANTITY': 'Cantidad',
    'ORDINAL': 'Ordinal', 'CARDINAL': 'Cardinal'
}


# 3. Análisis de sentimientos (Hugging Face)
print("\n" + "="*70)
print("Análisis de sentimientos (Hugging Face)")
print("="*70)

print(" Cargando modelo de sentimientos...")
# En Colab, esto descarga el modelo la primera vez (unos 500MB)
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)
print(" Modelo cargado correctamente")

textos_sentimiento = [
    "¡Me encanta este producto! Es increíble y funciona perfecto.",
    "Pésimo servicio, muy mala atención y el producto llegó roto.",
    "La película estaba bien, ni buena ni mala, totalmente promedio.",
    "Excelente experiencia, volvería a comprar sin dudarlo ni un segundo.",
    "No recomiendo para nada, perdí tiempo y dinero en esta compra."
]

print(f"\n Analizando {len(textos_sentimiento)} textos...\n")

resultados_sentimiento = []
for texto in textos_sentimiento:
    resultado = sentiment_pipeline(texto)[0]
    label = resultado['label']
    score = resultado['score']

    if '5' in label or '4' in label:
        sentimiento, color = "POSITIVO 😊", "green"
    elif '2' in label or '1' in label:
        sentimiento, color = "NEGATIVO 😞", "red"
    else:
        sentimiento, color = "NEUTRAL 😐", "orange"

    resultados_sentimiento.append({'texto': texto[:45] + "...", 'sentimiento': sentimiento, 'score': score, 'color': color})
    print(f" '{texto[:55]}...' \n   → {sentimiento} (Confianza: {score:.2%})\n")

# Visualizar resultados
fig, ax = plt.subplots(figsize=(12, 6))
labels = [r['sentimiento'] for r in resultados_sentimiento]
scores = [r['score'] for r in resultados_sentimiento]
colors = [r['color'] for r in resultados_sentimiento]

bars = ax.barh(range(len(labels)), scores, color=colors, edgecolor='black')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels([r['texto'] for r in resultados_sentimiento], fontsize=10)
ax.set_xlabel('Confianza del modelo', fontsize=12)
ax.set_title('😊 Análisis de Sentimientos con Hugging Face', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1)
ax.grid(axis='x', alpha=0.3)

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, f'{score:.2%}', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()


# 4. Ner Avanzado
print("\n" + "="*70)
print(" NER Avanzado (Extracción de Entidades)")
print("="*70)

texto_ner = "El presidente Pedro Sánchez se reunió con Emmanuel Macron en París el 15 de octubre de 2026. Apple y Microsoft anunciaron una inversión de 5000 millones de euros en España."
doc_ner = nlp_es(texto_ner)

print(f"\n Texto: '{texto_ner}'\n")
print(f"{'Entidad':<20} {'Tipo':<10} {'Descripción'}")
print("-" * 50)

ner_data = []
for ent in doc_ner.ents:
    ner_data.append({'entidad': ent.text, 'tipo': ent.label_, 'descripcion': NER_DESCRIPTIONS.get(ent.label_, ent.label_)})
    print(f"{ent.text:<20} {ent.label_:<10} {ner_data[-1]['descripcion']}")


# 5. Topic Modeling con scikit-learn (LDA)
print("\n" + "="*70)
print(" Topic Modeling (LDA con Scikit-Learn)")
print("="*70)

documentos = [
    "el presidente anunció nuevas medidas económicas para combatir la inflación",
    "el banco central subió las tasas de interés para controlar la economía",
    "la bolsa de valores cerró con ganancias récord este trimestre",
    "el equipo de fútbol ganó el campeonato nacional con gran actuación",
    "messi marcó tres goles en el partido contra brasil",
    "el entrenador del equipo fue despedido tras mala racha",
    "nueva tecnología de inteligencia artificial revoluciona la industria",
    "google presentó su nuevo modelo de lenguaje con capacidades avanzadas",
    "la startup tecnológica recibió financiación millonaria de inversores"
]

print(f"\n Tenemos {len(documentos)} documentos. Descubriendo 3 temas...\n")

# Preprocesamiento
def preprocess(text):
    doc = nlp_es(text.lower())
    return " ".join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha and len(token.lemma_) > 3])

documentos_procesados = [preprocess(doc) for doc in documentos]

# Vectorización y LDA
vectorizer = CountVectorizer(max_df=0.95, min_df=1)
X = vectorizer.fit_transform(documentos_procesados)

lda = LatentDirichletAllocation(n_components=3, random_state=42, max_iter=10)
lda.fit(X)

print("Temas descubiertos:\n")
feature_names = vectorizer.get_feature_names_out()
for idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-6:-1]]
    print(f"🔹 Tema {idx + 1}: {', '.join(top_words)}")


# 6. Lecciones Clave
print("\n" + "="*70)
print("💡 Lecciones clave del dia 23")
print("="*70)
print("""
1. Hugging Face: Permite usar modelos de IA de última generación con 1 línea de código.
2. NER (spaCy): Extrae información estructurada (personas, lugares, dinero) de texto crudo.
3. LDA (Scikit-Learn): Descubre temas ocultos en documentos sin necesidad de etiquetas previas.
4. Google Colab: Es el entorno ideal para NLP y Deep Learning. Evita errores de compilación
   de Windows (como los de Gensim/Torch en Python 3.14) y ofrece GPU gratuita.

CONCLUSIÓN: Ya puedes analizar texto, detectar sentimientos y extraer temas
como un profesional, sin depender de instalaciones locales complejas.
""")

print("\n✅ ¡Día 23 completado exitosamente en Colab! ")